In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

/home/jonathan/projects/primaite/PrimAITE/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class LLM:
    def __init__(self, model='HuggingFaceTB/SmolLM-1.7B-Instruct', device='cuda:1'):
        self.device = device
        self.tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
        self.model = GPT2LMHeadModel.from_pretrained('gpt2')
        
    def generate(self, prompt) -> str:
        """Your standard .generate"""
        
        ## Prepare prompt with template
        messages = [{"role": "user", "content": prompt}]
        inputs=self.tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt").to(self.device)
        output = self.model.generate(**inputs, max_new_tokens=69)
        return self.tokenizer.decode(output[0])

In [3]:
llm = LLM(device='cpu')

In [4]:
from soft_embedding import SoftEmbedding
n_tokens = 2048
initialize_from_vocab = True
s_wte = SoftEmbedding(wte=llm.model.get_input_embeddings(), n_tokens=n_tokens, initialize_from_vocab=initialize_from_vocab, n_prompts=1)

In [5]:
llm.model.vocab_size

49152

In [6]:
llm.model.set_input_embeddings(s_wte)

In [7]:
llm.tokenizer.pad_token_id

2

In [8]:
llm.tokenizer.model_max_length

2048

In [9]:
inputs = llm.tokenizer.encode_plus("Hello world! How a", return_tensors='pt', padding=True)
inputs['input_ids'] = torch.cat([inputs['input_ids'], torch.full((1,n_tokens), 2)], 1).to('cpu')
inputs['attention_mask'] = torch.cat([inputs['attention_mask'], torch.full((1,n_tokens), 0)], 1).to('cpu')

In [10]:
inputs.attention_mask

tensor([[1, 1, 1,  ..., 0, 0, 0]])

In [11]:
output = llm.model(**inputs)

IndexError: index out of range in self

In [ ]:
token_ids = torch.argmax(output.logits, dim=-1)

In [ ]:
llm.tokenizer.decode(token_ids[0])

'\n\n\n\n\n'

In [ ]:
output.logits.shape

torch.Size([1, 5, 49152])

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
llm = LLM(device='cuda:1')

In [ ]:
messages = [{"role": 'system', "content": "You are a helpful assistant",}, {"role": "user", "content": 'Say hello!'}]
prompt=llm.tokenizer.apply_chat_template(messages, tokenize=True, padding=True, return_tensors="pt").to('cuda:2')
with torch.no_grad():
    text_embeddings = llm.model.get_input_embeddings()(prompt)

RuntimeError: CUDA error: invalid device ordinal
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
print(llm.tokenizer.decode(prompt[0]))

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Say hello!<|im_end|>



In [ ]:
text_embeddings

tensor([[[-0.0442,  0.0498, -0.0293,  ..., -0.0131, -0.0112,  0.0036],
         [-0.0262, -0.0593, -0.0610,  ...,  0.0771,  0.0752, -0.0325],
         [-0.0737,  0.0304,  0.1465,  ..., -0.0535, -0.0170, -0.0311],
         ...,
         [ 0.1768,  0.0229,  0.0161,  ..., -0.1040, -0.0366, -0.0459],
         [-0.0277,  0.0238, -0.0135,  ..., -0.0208, -0.0128, -0.0099],
         [-0.0737,  0.0304,  0.1465,  ..., -0.0535, -0.0170, -0.0311]]],
       device='cuda:1', dtype=torch.float16)

In [ ]:
output_embeddings = llm.model.forward(inputs_embeds=text_embeddings)

In [ ]:
output_embeddings.logits.shape

torch.Size([1, 18, 49152])

In [ ]:
pred_token_ids = torch.argmax(output_embeddings.logits, dim=-1)
pred_token_ids

tensor([[ 9690,   198,     2,   359,  1836,  3197, 11173,   327,   198,     1,
           520,   198,  2020,   339,   288,   339,   198,     1]],
       device='cuda:1')

In [ ]:
pred_token_ids[0]

tensor([ 9690,   198,     2,   359,  1836,  3197, 11173,   327,   198,     1,
          520,   198,  2020,   339,   288,   339,   198,     1],
       device='cuda:1')

In [ ]:
llm.tokenizer.decode(pred_token_ids[0])

'system\n<|im_end|> are given software assistant for\n<|im_start|>ass\nHow I to I\n<|im_start|>'